<a href="https://colab.research.google.com/github/venkatasai-eng/MLA0305-REINFORCEMENT-LEARNING-/blob/main/EXP_NO_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

n_states = 5
n_actions = 2
episodes = 200
gamma = 0.9
learning_rate = 0.01

def step(state, action):
    if action == 1:
        next_state = min(state + 1, 4)
    else:
        next_state = max(state - 1, 0)

    reward = 10 if next_state == 4 else -1
    done = next_state == 4

    return next_state, reward, done

def create_policy():
    model = tf.keras.Sequential([
        layers.Input(shape=(n_states,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(n_actions, activation="softmax")
    ])
    return model

policy = create_policy()

optimizer = tf.keras.optimizers.Adam(
    learning_rate=learning_rate
)

for episode in range(episodes):

    state = 0
    states = []
    actions = []
    rewards = []

    while True:

        state_vector = np.zeros(n_states)
        state_vector[state] = 1

        probabilities = policy(
            state_vector.reshape(1, -1)
        )[0].numpy()

        action = np.random.choice(
            n_actions,
            p=probabilities
        )

        next_state, reward, done = step(
            state,
            action
        )

        states.append(state_vector)
        actions.append(action)
        rewards.append(reward)

        state = next_state

        if done:
            break

    returns = []
    G = 0

    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)

    returns = np.array(returns, dtype=np.float32)

    with tf.GradientTape() as tape:

        state_tensor = tf.convert_to_tensor(
            np.array(states),
            dtype=tf.float32
        )

        action_tensor = tf.convert_to_tensor(
            actions,
            dtype=tf.int32
        )

        probabilities = policy(state_tensor)

        selected_probabilities = tf.gather(
            probabilities,
            action_tensor,
            axis=1,
            batch_dims=1
        )

        log_probabilities = tf.math.log(
            selected_probabilities + 1e-8
        )

        loss = -tf.reduce_mean(
            log_probabilities * returns
        )

    gradients = tape.gradient(
        loss,
        policy.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, policy.trainable_variables)
    )

    if (episode + 1) % 20 == 0:
        print(
            "Episode:",
            episode + 1,
            "Total Reward:",
            sum(rewards)
        )

print("\nTraining completed successfully.")

print("\nLearned Policy:")

for state in range(n_states):

    state_vector = np.zeros(n_states)
    state_vector[state] = 1

    probabilities = policy(
        state_vector.reshape(1, -1)
    )[0].numpy()

    action = np.argmax(probabilities)

    print(
        "State",
        state,
        "-> Action",
        action,
        "Probabilities:",
        np.round(probabilities, 3)
    )

Episode: 20 Total Reward: 7
Episode: 40 Total Reward: 7
Episode: 60 Total Reward: 7
Episode: 80 Total Reward: 7
Episode: 100 Total Reward: 7
Episode: 120 Total Reward: 7
Episode: 140 Total Reward: 7
Episode: 160 Total Reward: 7
Episode: 180 Total Reward: 7
Episode: 200 Total Reward: 7

Training completed successfully.

Learned Policy:
State 0 -> Action 1 Probabilities: [0. 1.]
State 1 -> Action 1 Probabilities: [0. 1.]
State 2 -> Action 1 Probabilities: [0. 1.]
State 3 -> Action 1 Probabilities: [0. 1.]
State 4 -> Action 1 Probabilities: [0.002 0.998]
